# Use case 2: start NMHS forecasts at IFS step 60 with an Aviso `step` notification

**Ask (Marcos Bentos, Emma Kuwertz):** National Meteorological and Hydrological Services
(NMHSs) run their own limited-area forecasts from IFS boundary conditions. Today many wait
until the IFS run is "complete enough" (step 96 has been disseminated) before they start. If a
notification were sent as soon as **step 60** is available they could start earlier and gain
the difference in wall-clock time. That only works if the notification schema carries `step`.

This notebook shows that it does, builds the subscriber filter, demonstrates the
server-side step filtering and the triggers that would kick off the NMHS forecast.

| Setting | Value |
| --- | --- |
| Servers | Aviso 2 playground `https://aviso-playground.ecmwf.int` (`mars`, `dissemination`); LUMI data bridge `https://aviso2.lumi.apps.dte.destination-earth.eu` (`data`, read-only cross-check) |
| Auth | playground: ECMWF token (`AVISO_TOKEN` or `~/.config/aviso/config.yaml`); LUMI: DESP offline token (`~/.polytopeapirc`) |
| Client | `pip install pyaviso pyyaml pandas` (pyaviso 2.1.0 at the time of writing) |

Sections:

1. Setup, authentication and helpers
2. The schemas carry `step`
3. From "wait for step 96" to "act on step 60"
4. The subscriber filter: exact value and constraints
5. Server-side filtering, demonstrated live
6. Triggers that start the NMHS forecast; catch-up and resume
7. Cross-check on the LUMI data bridge (Extremes-DT step 60)
8. Notes and open points

## 1. Setup, authentication and helpers

The playground needs an ECMWF token (from <https://api.ecmwf.int/v1/key/>), the LUMI server a
DESP offline token. Tokens are read from the environment or the usual files and never printed.

In [1]:
# %pip install -q pyaviso pyyaml pandas
import json, os, pathlib, queue, threading, time
from datetime import datetime, timedelta, timezone

import pandas as pd
import pyaviso

print("pyaviso", pyaviso.__version__)

PLAYGROUND = "https://aviso-playground.ecmwf.int"
LUMI = "https://aviso2.lumi.apps.dte.destination-earth.eu"
OUT_DIR = pathlib.Path("output") / "uc02"
OUT_DIR.mkdir(parents=True, exist_ok=True)


def ecmwf_token():
    tok = os.environ.get("AVISO_TOKEN")
    if tok:
        return tok
    cfg = pathlib.Path.home() / ".config" / "aviso" / "config.yaml"
    if cfg.exists():
        import yaml
        tok = ((yaml.safe_load(cfg.read_text()) or {}).get("auth") or {}).get("bearer_token")
    if not tok:
        raise RuntimeError("set AVISO_TOKEN or auth.bearer_token in ~/.config/aviso/config.yaml")
    return tok


def desp_token():
    rc = pathlib.Path.home() / ".polytopeapirc"
    if rc.exists():
        return json.loads(rc.read_text()).get("user_key")
    return None


def open_stream(client, event_type, *, filter, start=None, mode=None, triggers=None):
    # pyaviso 2.1.0 names the cursor `start_from=`; newer builds `from_=`.
    kwargs = {"filter": filter}
    if mode is not None:
        kwargs["mode"] = mode
    if triggers is not None:
        kwargs["triggers"] = triggers
    if start is None:
        return client.listen(event_type, **kwargs)
    try:
        return client.listen(event_type, start_from=start, **kwargs)
    except TypeError:
        return client.listen(event_type, from_=start, **kwargs)


def replay(client, event_type, *, filter, start=1, max_pages=None):
    # Page through a replay; servers cap one request (LUMI: 10k) and raise HistoryGapError.
    out, cursor, pages = [], start, 0
    while True:
        pages += 1
        try:
            with open_stream(client, event_type, filter=filter, start=cursor, mode="replay_only") as s:
                out.extend(s)
            return out
        except pyaviso.HistoryGapError as exc:
            if "replay_limit" not in str(exc) or not out or out[-1].sequence == cursor:
                raise
            cursor = out[-1].sequence
            if max_pages and pages >= max_pages:
                return out


def has_after(client, event_type, seq, filter):
    with open_stream(client, event_type, filter=filter, start=seq, mode="replay_only") as s:
        return next(iter(s), None) is not None


def latest_sequence(client, event_type, filter):
    lo, hi = 0, 1
    while has_after(client, event_type, hi, filter):
        lo, hi = hi, hi * 2
    while hi - lo > 1:
        mid = (lo + hi) // 2
        if has_after(client, event_type, mid, filter):
            lo = mid
        else:
            hi = mid
    return lo + 1


def replay_tail(client, event_type, *, filter, probe_filter, span=3000):
    last = latest_sequence(client, event_type, probe_filter)
    return replay(client, event_type, filter=filter, start=max(last - span, 0))


def listen_collect(client, event_type, *, filter, seconds=20, max_events=None, start=None, on_event=None, triggers=None):
    # Worker thread blocks on the stream; this thread applies the deadline and handles events.
    stream = open_stream(client, event_type, filter=filter, start=start, triggers=triggers)
    q = queue.Queue()

    def pump():
        try:
            for n in stream:
                q.put(n)
        except Exception as exc:
            q.put(exc)
        finally:
            q.put(None)

    threading.Thread(target=pump, daemon=True).start()
    got, deadline, t0 = [], time.monotonic() + seconds, time.monotonic()
    while time.monotonic() < deadline:
        try:
            item = q.get(timeout=min(1.0, max(deadline - time.monotonic(), 0.01)))
        except queue.Empty:
            continue
        if item is None:
            break
        if isinstance(item, Exception):
            print("stream error:", type(item).__name__, item)
            break
        got.append(item)
        print(f"[{time.monotonic() - t0:5.1f}s] seq={item.sequence} {item.identifier} payload={item.payload}")
        if on_event:
            on_event(item)
        if max_events and len(got) >= max_events:
            break
    stream.close()
    print(f"listener closed after {time.monotonic() - t0:.1f}s with {len(got)} notification(s)")
    return got


pg = pyaviso.AvisoClient(base_url=PLAYGROUND, auth=pyaviso.Bearer(ecmwf_token()))
print("playground event types:", sorted(pg.schema().event_types))

pyaviso 2.1.0
playground event types: ['cleareo_app1_data_available', 'cleareo_app1_extreme_rainfall', 'de-scm-data', 'de-scm-data-points', 'dissemination', 'extreme_event', 'mars']


## 2. The schemas carry `step`

The prerequisite of the use case is a `step` identifier. Both ECMWF product streams on the
Aviso 2 playground have it as an `IntHandler`, and so does the DestinE `data` stream on the
LUMI data bridge. Aviso 1.x already carried `step` in its `mars` and `dissemination`
listeners (`aviso-deployment/schema/Bologna/aviso-prod/event_listener_schema.json`), so this
is not new information for the producers, only a new way to subscribe to it.

In [2]:
rows = []
for base, events in ((PLAYGROUND, ["mars", "dissemination"]), (LUMI, ["data"])):
    anon = pyaviso.AvisoClient(base_url=base)           # the schema listing is public on both servers
    for ev in events:
        ident = anon.schema_for(ev).schema["identifier"]
        for key, spec in ident.items():
            rows.append({"server": base.split("//")[1].split(".")[0], "event": ev, "identifier": key,
                         "type": spec.get("type"), "required": spec.get("required", False),
                         "range / values": spec.get("range") or spec.get("values") or spec.get("canonical_format") or ""})
schemas = pd.DataFrame(rows)
schemas[schemas["identifier"] == "step"]

,server,event,identifier,type,required,range / values
4,aviso-playground,mars,step,IntHandler,False,"[0, 100000]"
12,aviso-playground,dissemination,step,IntHandler,False,"[0, 100000]"
21,aviso2,data,step,IntHandler,False,"[0, 10000]"


In [3]:
schemas[schemas["event"] == "mars"].set_index("identifier")[["type", "required", "range / values"]]

,type,required,range / values
identifier,,,
class,StringHandler,True,
date,DateHandler,False,%Y%m%d
domain,EnumHandler,False,"[a, b, c, d, e, f, g, h, i, j, k, l, m, n, o, ..."
expver,ExpverHandler,False,
step,IntHandler,False,"[0, 100000]"
stream,StringHandler,False,
time,TimeHandler,False,


## 3. From "wait for step 96" to "act on step 60"

IFS HRES products are disseminated step by step as the model integrates. An NMHS that needs
boundary conditions up to T+60 only has to wait for the T+60 products, but without a per-step
signal the safe strategy has been to wait for a late marker such as step 96. With Aviso the
producer (MARS archiving, or ECPDS dissemination to the NMHS destination) publishes one
notification per step, and the subscriber filters on exactly the step it needs.

```
IFS run 00Z  ──► step 0 … step 57, step 60 ──────────────► … step 96
                                    │                              │
                        Aviso: mars/dissemination step=60     (old trigger)
                                    ▼
                        NMHS LAM forecast starts here
```

The gain is the dissemination time between step 60 and step 96 of the same run, every cycle.

## 4. The subscriber filter: exact value and constraints

An Aviso 2 filter is the set of identifiers you commit to; omitted optional keys are wildcards.
`class` is the only required key on `mars`; `dissemination` also requires `destination` (the
ECPDS destination of the NMHS, which the server authorises through ECPDS). `IntHandler` fields
accept a scalar or a constraint object (`eq`, `in`, `gt`, `gte`, `lt`, `lte`, `between`).

Aviso 1.x listener for comparison (from the 1.x `mars` schema; keys are identical):

```yaml
listeners:
  - event: mars
    request: {class: od, expver: 1, domain: g, stream: oper, step: 60}
    triggers: [{type: echo}]
```

In [4]:
NMHS_STEP = 60
FILTERS = {
    "mars, exactly step 60":            {"class": "od", "expver": "0001", "stream": "oper", "step": NMHS_STEP},
    "mars, step 60 or later":           {"class": "od", "expver": "0001", "stream": "oper", "step": {"gte": NMHS_STEP}},
    "mars, any step in 60..72":         {"class": "od", "expver": "0001", "stream": "oper", "step": {"between": [60, 72]}},
    "mars, one of several steps":       {"class": "od", "expver": "0001", "stream": "oper", "step": {"in": [60, 66, 72]}},
    "mars, 00Z run only":               {"class": "od", "expver": "0001", "stream": "oper", "time": "0000", "step": NMHS_STEP},
    "dissemination, NMHS destination":  {"class": "od", "destination": "*", "stream": "oper", "step": NMHS_STEP},
}

# Validate each filter against the live server with a replay (accepted = the server parsed it).
for label, f in FILTERS.items():
    ev = label.split(",")[0]
    try:
        n = len(replay(pg, ev, filter=f, start=1, max_pages=1))
        print(f"accepted  {label:<36} {json.dumps(f)}  -> {n} retained notification(s)")
    except pyaviso.HttpError as exc:
        body = json.loads(exc.body) if exc.body else {}
        print(f"HTTP {exc.status:<4} {label:<36} {body.get('code')}: {body.get('message')}")

accepted  mars, exactly step 60                {"class": "od", "expver": "0001", "stream": "oper", "step": 60}  -> 0 retained notification(s)
accepted  mars, step 60 or later               {"class": "od", "expver": "0001", "stream": "oper", "step": {"gte": 60}}  -> 0 retained notification(s)
accepted  mars, any step in 60..72             {"class": "od", "expver": "0001", "stream": "oper", "step": {"between": [60, 72]}}  -> 0 retained notification(s)
accepted  mars, one of several steps           {"class": "od", "expver": "0001", "stream": "oper", "step": {"in": [60, 66, 72]}}  -> 0 retained notification(s)
accepted  mars, 00Z run only                   {"class": "od", "expver": "0001", "stream": "oper", "time": "0000", "step": 60}  -> 0 retained notification(s)


HTTP 403  dissemination, NMHS destination      FORBIDDEN: User does not have required read role for this stream


The `dissemination` replay is refused with **403**: on this deployment the stream is guarded by
the ECPDS destination plugin, so only a user who owns the destination gets its notifications.
That is exactly the behaviour an NMHS wants for its own dissemination stream. The `mars`
filters are all accepted; the stream is empty on the playground, so nothing is replayed.

To make the matching rule tangible without a producer, the cell below applies the same
semantics to a synthetic run of notifications built with `pyaviso.Notification`.

In [5]:
def matches(filter, identifier):
    # Local mirror of the server rule: every filter key must match; constraint objects on numbers.
    for key, want in filter.items():
        have = identifier.get(key)
        if want == "*":
            continue
        if isinstance(want, dict):
            v = float(have)
            ok = all({
                "eq": lambda a: v == a, "gt": lambda a: v > a, "gte": lambda a: v >= a,
                "lt": lambda a: v < a, "lte": lambda a: v <= a,
                "in": lambda a: v in a, "between": lambda a: a[0] <= v <= a[1],
            }[op](arg) for op, arg in want.items())
            if not ok:
                return False
        elif str(have) != str(want):
            return False
    return True


today = datetime.now(timezone.utc).strftime("%Y%m%d")
run = [pyaviso.Notification("mars", i + 1,
                            {"class": "od", "expver": "0001", "domain": "g", "stream": "oper",
                             "date": today, "time": "0000", "step": str(step)}, None)
       for i, step in enumerate(range(0, 97, 3))]

table = pd.DataFrame({"step": [int(n.identifier["step"]) for n in run]})
for label, f in FILTERS.items():
    if label.startswith("mars"):
        table[label.split(", ", 1)[1]] = [matches(f, n.identifier) for n in run]
table.set_index("step").T.replace({True: "x", False: ""})

step,0,3,6,9,12,15,18,21,24,27,...,69,72,75,78,81,84,87,90,93,96
exactly step 60,,,,,,,,,,,...,,,,,,,,,,
step 60 or later,,,,,,,,,,,...,x,x,x,x,x,x,x,x,x,x
any step in 60..72,,,,,,,,,,,...,x,x,,,,,,,,
one of several steps,,,,,,,,,,,...,,x,,,,,,,,
00Z run only,,,,,,,,,,,...,,,,,,,,,,


## 5. Server-side filtering, demonstrated live

The real proof is a live watch on the server. Two things about the playground shape this demo:

* Publishing to `mars` is refused for ordinary users (write role missing, HTTP 403, checked in
  the cell below). Only the archiving system may announce MARS data, which is as it should be.
* `extreme_event` is writable by any token holder and has an `IntHandler` (`severity`, 1..7)
  that behaves exactly like `step`. It is therefore used as a **stand-in** for the mechanics:
  a listener with `severity >= 5` is started, then a low and a high severity event are
  published from a second thread, and only the high one arrives.

The subscriber code is identical to the `mars`/`dissemination` case; only the event type and
field names differ.

In [6]:
TRY_PUBLISH_MARS = True   # one deliberately attempted publish, to record the server's answer
if TRY_PUBLISH_MARS:
    try:
        r = pg.notify(event_type="mars",
                      identifier={"class": "od", "expver": "0099", "domain": "g", "stream": "oper",
                                  "date": today, "time": "0000", "step": str(NMHS_STEP)},
                      payload={"note": "nmhs-uc2-notebook-test"})
        print("mars publish accepted:", r.status, r.request_id)
    except pyaviso.HttpError as exc:
        body = json.loads(exc.body) if exc.body else {}
        print(f"mars publish refused: HTTP {exc.status} {body.get('code')}: {body.get('message')}")

mars publish refused: HTTP 403 FORBIDDEN: User does not have required write role for this stream


In [7]:
# Stand-in live demo on `extreme_event` (writable on the playground). severity plays the role of step.
berlin = "(52.5,13.4,52.6,13.5,52.5,13.6,52.4,13.5,52.5,13.4)"
STAND_IN_FILTER = {"region": "north", "run_time": "1200", "severity": {"gte": 5}, "point": "52.55,13.50"}
TAG = f"uc2-step60-demo-{datetime.now(timezone.utc):%H%M%S}"


def publish_pair():
    time.sleep(3)                              # let the SSE handshake finish first
    for sev in ("2", "6"):                     # 2 must be filtered out, 6 must arrive
        try:
            r = pg.notify(event_type="extreme_event",
                          identifier={"region": "north", "run_time": "1200", "severity": sev,
                                      "anomaly": "42.5", "polygon": berlin},
                          payload={"note": TAG, "severity_published": sev})
            print(f"   published severity={sev}: {r.status}")
        except pyaviso.HttpError as exc:
            print(f"   publish severity={sev} refused: HTTP {exc.status}")
        time.sleep(1)


threading.Thread(target=publish_pair, daemon=True).start()
arrived = listen_collect(pg, "extreme_event", filter=STAND_IN_FILTER, seconds=20, max_events=1)
assert all(n.payload.get("severity_published") == "6" for n in arrived if n.payload and n.payload.get("note") == TAG), "a filtered-out event slipped through"
print("only the event satisfying the constraint was delivered" if arrived else "nothing arrived (check the publish output above)")

   published severity=2: success


[  4.1s] seq=41 {'anomaly': '42.5', 'polygon': [[52.5, 13.4], [52.6, 13.5], [52.5, 13.6], [52.4, 13.5], [52.5, 13.4]], 'region': 'north', 'run_time': '1200', 'severity': '6'} payload={'note': 'uc2-step60-demo-153707', 'severity_published': '6'}   published severity=6: success

listener closed after 4.2s with 1 notification(s)
only the event satisfying the constraint was delivered


## 6. Triggers that start the NMHS forecast; catch-up and resume

Once the step-60 notification arrives something has to run. Aviso 2 triggers execute per
notification **before** the notification is yielded to your loop, so they work in the CLI and
in Python alike. For an NMHS the obvious choices are:

* `command`: start the LAM suite (an ecFlow/`ecflow_client --alter`, a `sbatch`, a script).
  Fields are available as `{{ notification.<path> }}` templates and as environment variables.
* `webhook` / `post`: hand the CloudEvent to a workflow manager's HTTP API.
* `log`: keep an NDJSON audit trail of what triggered when.

Below, a `command` trigger writes a marker file with the run's date/time/step, and a `log`
trigger records the notification; both are exercised through a replay of the stand-in stream
so the mechanics are visible without a live producer.

In [8]:
marker = OUT_DIR / "start-nmhs-forecast.txt"
audit = OUT_DIR / "step60-events.ndjson"
marker.unlink(missing_ok=True); audit.unlink(missing_ok=True)

start_cmd = pyaviso.Trigger.command(
    'echo "START LAM for run {{ notification.identifier.run_time }}Z (severity={{ notification.identifier.severity }}) seq={{ notification.sequence }}" >> ' + str(marker.resolve()),
    timeout=10.0,
)
# For the real stream the template would read:
#   ... "{{ notification.identifier.date }} {{ notification.identifier.time }} step {{ notification.identifier.step }}" ...

with open_stream(pg, "extreme_event", filter=STAND_IN_FILTER, start=1, mode="replay_only",
                 triggers=[start_cmd, pyaviso.Trigger.log(audit)]) as s:
    replayed = list(s)
print(f"{len(replayed)} matching notification(s) replayed; marker file:")
print(marker.read_text() if marker.exists() else "(no marker written)")
print("audit lines:", sum(1 for _ in audit.open()) if audit.exists() else 0)

2 matching notification(s) replayed; marker file:
START LAM for run 1200Z (severity=6) seq=39
START LAM for run 1200Z (severity=6) seq=41

audit lines: 2


### CLI listener for production

```yaml
# nmhs-step60.yaml
listeners:
  - name: ifs-step60
    event: mars                      # or `dissemination` with `destination: <your ECPDS destination>`
    identifiers:
      class: od
      expver: "0001"
      stream: oper
      step: 60
    triggers:
      - type: log
        path: /var/log/aviso/ifs-step60.ndjson
      - type: command
        command: >-
          /opt/nmhs/bin/start_lam.sh --date {{ notification.identifier.date }}
          --time {{ notification.identifier.time }} --step {{ notification.identifier.step }}
        timeout: 120
        retries: 2
```

```bash
export AVISO_BASE_URL=https://aviso-playground.ecmwf.int   # production URL once available
export AVISO_TOKEN=...                                       # ECMWF API key
aviso listen nmhs-step60.yaml                                # live from now
aviso listen nmhs-step60.yaml --from 2026-09-24T00:00:00Z    # replay missed runs first, then stay live
```

The CLI keeps a cursor per (server, event, filter) in `~/.config/aviso/state.json`, so a
restart resumes where it stopped. The Python equivalent is a `JsonFileStore`:

In [9]:
state = OUT_DIR / "state.json"
resuming = pyaviso.AvisoClient(base_url=PLAYGROUND, auth=pyaviso.Bearer(ecmwf_token()),
                               state_store=pyaviso.JsonFileStore(state), flush_cursor_on_exit=True)
_ = listen_collect(resuming, "extreme_event", filter=STAND_IN_FILTER, seconds=5, start=1)
print(json.dumps(json.loads(state.read_text()), indent=2) if state.exists() else "no state written")

[  0.1s] seq=39 {'anomaly': '42.5', 'polygon': [[52.5, 13.4], [52.6, 13.5], [52.5, 13.6], [52.4, 13.5], [52.5, 13.4]], 'region': 'north', 'run_time': '1200', 'severity': '6'} payload={'note': 'uc2-step60-demo-153603', 'severity_published': '6'}
[  0.1s] seq=41 {'anomaly': '42.5', 'polygon': [[52.5, 13.4], [52.6, 13.5], [52.5, 13.6], [52.4, 13.5], [52.5, 13.4]], 'region': 'north', 'run_time': '1200', 'severity': '6'} payload={'note': 'uc2-step60-demo-153707', 'severity_published': '6'}


listener closed after 5.1s with 2 notification(s)
{
  "version": 1,
  "key_format_version": 1,
  "checkpoints": {
    "5141940b8c9f41db0ee4345320a9212c0f5f77e1c1a1c4709beec0dece8895e5": {
      "last_committed_sequence": 41,
      "last_event_id": "extreme_event@41"
    }
  }
}


## 7. Cross-check on the LUMI data bridge (Extremes-DT step 60)

The DestinE Aviso 2 server announces Extremes-DT fields per step on its `data` stream. The same
filter idea (`step: 60`) applied there returns the step-60 announcements for the newest
retained runs. Read-only: this notebook never publishes to LUMI. The stream held only a
back-fill published on 2026-09-16 when this was written, so dates are older than the runs.

In [10]:
tok = desp_token()
if not tok:
    print("no ~/.polytopeapirc found: skipping the LUMI cross-check")
else:
    lumi = pyaviso.AvisoClient(base_url=LUMI, auth=pyaviso.Bearer(tok))
    try:
        t0 = time.monotonic()
        step60 = replay_tail(lumi, "data",
                             filter={"class": "d1", "stream": "oper", "levtype": "sfc", "step": NMHS_STEP},
                             probe_filter={"class": "d1"}, span=6000)
        print(f"{len(step60)} step-{NMHS_STEP} sfc announcements in the last 6000 sequences ({time.monotonic() - t0:.1f}s)")
        rows = [{"sequence": n.sequence, **n.identifier, "published": (n.cloudevent or {}).get("time")} for n in step60]
        display(pd.DataFrame(rows).tail(8))
    except pyaviso.HttpError as exc:
        print("LUMI replay refused: HTTP", exc.status, exc.body[:200])

15 step-60 sfc announcements in the last 6000 sequences (11.4s)


,sequence,class,date,domain,expver,levtype,step,stream,time,type,published
7,136757,d1,20260908,g,0001,sfc,60,oper,0000,fc,2026-09-16T12:18:10.800039387Z
8,137150,d1,20260909,g,0001,sfc,60,oper,0000,fc,2026-09-16T12:18:11.105791479Z
9,137536,d1,20260910,g,0001,sfc,60,oper,0000,fc,2026-09-16T12:18:11.572268148Z
10,137935,d1,20260911,g,0001,sfc,60,oper,0000,fc,2026-09-16T12:18:11.942220732Z
11,138323,d1,20260912,g,0001,sfc,60,oper,0000,fc,2026-09-16T12:18:12.244144270Z
12,138700,d1,20260913,g,0001,sfc,60,oper,0000,fc,2026-09-16T12:18:12.758119117Z
13,139085,d1,20260914,g,0001,sfc,60,oper,0000,fc,2026-09-16T12:18:13.062029960Z
14,139469,d1,20260915,g,0001,sfc,60,oper,0000,fc,2026-09-16T12:18:13.600676651Z


## 8. Notes and open points

* **Schema is not the blocker.** `step` is an `IntHandler` on `mars`, `dissemination` and the
  DestinE `data` stream, with exact and range matching server-side. What the use case needs
  is the **producer** to publish one notification per step (MARS archiving and ECPDS
  dissemination did so in Aviso 1.x) on an Aviso 2 server the NMHSs can reach.
* **Which stream.** `dissemination` is the natural one for an NMHS: it fires when the products
  have landed at *their* destination, and the ECPDS plugin restricts reads to the destination
  owner (the 403 seen in section 4 with a token that owns no destination). `mars` fires on
  archiving, earlier but not tied to delivery.
* **Playground limits.** `mars` is read-only for normal tokens (403 on notify, section 5), and
  both product streams were empty on the playground on 2026-09-24, hence the `extreme_event`
  stand-in for the live demonstration.
* **Time value format.** Aviso 2 `TimeHandler` values are `HHMM` strings (`"0000"`), where
  1.x used `0`, `6`, `12`, `18`; adjust existing listener configurations accordingly.

## Related Polytope examples

The DestinE Polytope examples repository (`~/Git/polytope/polytope-examples`, GitHub
`destination-earth-digital-twins/polytope-examples`) contains no Aviso code, but it is the
reference for the **data side** of every Aviso workflow: once a notification names a field, these
notebooks show how to fetch exactly that field (or a feature cut from it) with earthkit-data.

| Need | Example |
| --- | --- |
| Full Extremes-DT field, regional `area` and `grid` options | `extremes-dt/full-field/extremes-dt-earthkit-example-domain.ipynb`, `extremes-dt/full-field-post-processing/extremes-dt-earthkit-serverside-interpolation.ipynb` |
| Feature extraction (timeseries, bounding box, polygon/country, vertical profile, trajectory, wave) | `extremes-dt/feature-extraction/*.ipynb` |
| On-Demand Extremes-DT (DEODE) output: `dataset: on-demand-extremes-dt`, `expver: 0099`, per-run `georef` | `on-demand-extremes-dt/full-field/` and `on-demand-extremes-dt/feature-extraction/` |
| DESP token (`~/.polytopeapirc`) | `desp-authentication.py` |

Operational limits from that README: 50 requests per second and at most 5 concurrent downloads per
user; data are served from LUMI (`polytope.lumi.apps.dte.destination-earth.eu`) and MN5
(`polytope.mn5.apps.dte.destination-earth.eu`). Relative dates such as `"date": "-14"` are accepted.
